**The weebs on Tik Tok made a funny joke so lets take it a step further...**

There is a website that allows you to take a 50 question autism test which gives you a nicely presented radar chart.

All was fine until people from the sizeble instersection between autism and JoJo fans noticed JoJo stands have stats also presented in radar charts.

I think the rest is pretty self explanatory...

<img alt="Example of an autism test result" src="https://charts.idrlabs.com/graphic/autism-spectrum?2&p=35,45,35,10,80,30,35,70,50,0&l=EN" width="300" /> <img alt="Example of a JoJo's stand stat summary" src="https://static.wikia.nocookie.net/f86e4024-c544-4860-801e-d9f4afd8e0e4/scale-to-width/755" width="250" />

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import re
from shapely.geometry import Polygon

To extract the results from the autism test is fairly simple. The url for the results image has the results in a numerical form as part of the url parameters, so its as simple as extracting the results from there :)

In [ ]:
# Image from the autism test result at https://www.idrlabs.com/autism-spectrum/test.php, replace this with your own result :)
autism_spectrum_image_url = "https://charts.idrlabs.com/graphic/autism-spectrum?2&p=35,45,35,10,80,30,35,70,50,0&l=EN"

# Extract the stats from the url
autism_stats = re.findall(".*p=([^&|\n|\t\s]+)", autism_spectrum_image_url)[0].split(",")
autism_stats = list(map(int, autism_stats))
print(autism_stats)

Now comes translating the JoJo stand stats into a numerical form so they can be easily compared. The values were originally None, E, D, C, B, A, Infi(nity), and they are being replaced by a respective value from 0 to 100 to make it easier to compare with the autism results.

In [ ]:
df = pd.read_csv("/kaggle/input/jojo-stands-stats/jojo-stands.csv", encoding='ISO-8859-1')

# Convert stats labels to numerical values from 0 to 100
df = df.replace(to_replace=['None', 'E', 'D', 'C', 'B', 'A', 'Infi'], value=[i * 100/6 for i in range(7)])
# Rename one of the column headers for my own sanity
df.columns = ['STM' if x=='PER' else x for x in df.columns]

df.head()

One issue is that the autism test results are 10 dimensional, having 10 different values to make up the whole result, whereas the JoJo stand stats are 6 dimensional. To combat this, I decided to take weighted averages of a few neighbouring stats on the radio chart based on my own arbitrary choice to reduce the 10 values to 6. This could be changed up to place more weights on different stats if you this it would be more appropriate.

In [ ]:
# Function to convert the autism results (10 dimensional) into stand stats (6 dimensional)
def autism_to_stand_stats(depression, fixation, speech, noise, social, anxiety, posture, eye, tics, aggression):
    pwr = (depression + fixation) / 2
    spd = (fixation + noise) / 4 + speech / 2
    rng = (anxiety + noise) / 4 + social / 2
    stm = (posture + anxiety) / 2
    prc = (tics + posture) / 4 + eye / 2
    dev = (tics + depression) / 4 + aggression / 2
    return np.array([pwr, spd, rng, stm, prc, dev])
    
converted_stats = autism_to_stand_stats(*autism_stats)
print(converted_stats)

To help visualise if the stat conversion is giving appropriate results, here are some functions to plot the stats in a radio chart similar to how they were originally presented.

In [ ]:
def plot_circles():
    for i in [i * 100/6 for i in range(7)]:
        c = plt.Circle((0, 0), i, color='g', linestyle="--", fill=False)
        plt.gca().add_patch(c)

def plot_stats(s):
    circle = [[-np.cos(np.pi/3 * t + np.pi/2), np.sin(np.pi/3 * t + np.pi/2)]
              for t in range(6)]
    coords = [np.array(circle[i]) * s[i] for i in range(6)]
    p = Polygon(coords)
    plt.plot(*p.exterior.xy)

# Visualise the converted stats in a JoJo's-like fashion
plot_circles()
plot_stats(converted_stats)

Now, to find the most similar charts, I approached this by finding the distances between the converted autism stats and all of the JoJo stand stats as it is fairly simple as they are represented by 6 dimensional coordinates

In [ ]:
def distance(a, b):
    try:
        return np.linalg.norm(a-b)
    except:
        print(a)
        print(b)
        raise

# Calculate the similarity for each stand compared to the autism stats
df["distance"] = df.apply(lambda x: distance(x.to_numpy()[1:7], converted_stats), axis=1)

df.head()

After calculating all of the distances, finding the most simlar JoJo stand is as simple as finding the minimum distance :)

In [ ]:
# Find the most similar stand
most_similar = df.loc[df['distance'].idxmin()]

plot_circles()
plot_stats(most_similar.to_numpy()[1:7])
plot_stats(converted_stats)

print(most_similar)